# 04 — Export artifacts (ONNX, vocab JSON, seed SQL, fixed test vector)

Produces everything the Java backend needs:

1. `classifier.onnx` — the calibrated linear classifier (input = the TF-IDF vector).
2. `vocab.json` — word+char vocabularies and IDF weights, so **Java rebuilds the TF-IDF vector**.
   *Why split?* `skl2onnx` cannot convert a char n-gram TF-IDF vectorizer (`char_wb` raises
   `NotImplementedError`), so the vectorizer stays in Java and only the classifier goes to ONNX.
3. `coef.json` — LR coefficients + intercept + Platt (a,b), for the `coef×tfidf` **log-odds**
   contributions and the calibrated probability.
4. `salary.onnx` + `salary_meta.json` — the salary regressor and residual σ.
5. `known_scams_seed.sql` — the 866 confirmed frauds embedded with all-MiniLM-L6-v2 (384-d).
6. **A fixed test vector and the exact output probability** — the number Java asserts against.

This notebook re-fits the served model deterministically (same seeds as 02) so it is
self-contained. **ONNX files are not committed to git.**

> **Kaggle setup:** Internet = On. GPU optional (speeds up the embedding step).

In [ ]:
!pip -q install kagglehub xgboost skl2onnx onnx onnxruntime onnxmltools sentence-transformers

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
SEED = 42

import kagglehub
DATA_DIR = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
csvs = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
pref = [f for f in csvs if "fake_job_postings" in os.path.basename(f).lower()]
CSV = pref[0] if pref else csvs[0]
df = pd.read_csv(CSV)
print("Loaded:", CSV)
print("Shape:", df.shape)

In [ ]:
# The product is fed pasted free text, so the served model uses ONLY text columns.
# Structured meta-features (has_company_logo, telecommuting, has_questions) are
# deliberately EXCLUDED: they are predictive in EMSCAD but unavailable/leaky at
# serve time, and using them would create train/serve skew.
TEXT_COLS = ["title", "company_profile", "description", "requirements", "benefits"]

def build_text(frame):
    present = [c for c in TEXT_COLS if c in frame.columns]
    return (frame[present].fillna("").astype(str).agg(" ".join, axis=1)
            .str.replace(r"\s+", " ", regex=True).str.strip())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

# EXACT TF-IDF config — Java must replicate this precisely (see 04 vocab.json export).
# Two blocks, each L2-normalised INDEPENDENTLY, then concatenated (word || char).
WORD_PARAMS = dict(analyzer="word", ngram_range=(1, 2), lowercase=True,
                   strip_accents="unicode", min_df=3, max_features=40000,
                   sublinear_tf=True, norm="l2", use_idf=True, smooth_idf=True)
CHAR_PARAMS = dict(analyzer="char_wb", ngram_range=(3, 5), lowercase=True,
                   strip_accents="unicode", min_df=3, max_features=40000,
                   sublinear_tf=True, norm="l2", use_idf=True, smooth_idf=True)

def build_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(**WORD_PARAMS)),
        ("char", TfidfVectorizer(**CHAR_PARAMS)),
    ])

In [ ]:
# --- Re-fit the served model exactly as in 02 (deterministic) ---
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

# Same dedup + seeds as 02, so the exported model matches the evaluated one.
_d = pd.DataFrame({"text": build_text(df).values, "y": df["fraudulent"].astype(int).values})
_d = _d.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
text, y = _d["text"].values, _d["y"].values
X_tr_text, X_te_text, y_tr, y_te = train_test_split(text, y, test_size=0.20, stratify=y, random_state=SEED)
X_fit_text, X_cal_text, y_fit, y_cal = train_test_split(X_tr_text, y_tr, test_size=0.25, stratify=y_tr, random_state=SEED)

vectorizer = build_vectorizer()
Xf = vectorizer.fit_transform(X_fit_text)
Xc = vectorizer.transform(X_cal_text)
lr = LogisticRegression(class_weight="balanced", solver="liblinear", C=1.0, max_iter=2000, random_state=SEED).fit(Xf, y_fit)
try:
    from sklearn.frozen import FrozenEstimator      # sklearn >= 1.6
    served = CalibratedClassifierCV(FrozenEstimator(lr), method="sigmoid").fit(Xc, y_cal)
except ImportError:
    served = CalibratedClassifierCV(lr, method="sigmoid", cv="prefit").fit(Xc, y_cal)
n_features = Xf.shape[1]
print("Re-fit complete. n_features =", n_features)

In [ ]:
# --- Export the classifier to ONNX (try calibrated; fall back to raw LR) ---
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [("input", FloatTensorType([None, n_features]))]
export_kind, onx = None, None
try:
    onx = convert_sklearn(served, initial_types=initial_type, options={id(served): {"zipmap": False}})
    export_kind = "calibrated"
except Exception as e:
    # Expected when `served` wraps a FrozenEstimator (no skl2onnx converter) — fall back
    # to the raw LR; coef.json carries the Platt (a,b) so Java still gets the calibrated prob.
    print("Calibrated export unavailable; exporting raw LR instead:", repr(e))
    try:
        onx = convert_sklearn(lr, initial_types=initial_type, options={id(lr): {"zipmap": False}})
        export_kind = "raw_lr"
    except Exception as e2:
        print("Raw LR ONNX export also failed:", repr(e2))

classifier_onnx_ok = onx is not None
if classifier_onnx_ok:
    with open(os.path.join(OUT, "classifier.onnx"), "wb") as f:
        f.write(onx.SerializeToString())
    print("Wrote classifier.onnx  (export_kind =", export_kind, ")")
else:
    print("No classifier.onnx written; Java can still serve from coef.json (coef + intercept + Platt).")

In [ ]:
# --- Extract Platt (a, b) calibration params ---
cc = served.calibrated_classifiers_[0]
sig = getattr(cc, "calibrators", None) or getattr(cc, "calibrators_", None)
sig = sig[0]
platt_a, platt_b = float(sig.a_), float(sig.b_)
# sklearn: calibrated_p = expit(-(a * f + b)), where f = decision_function (the logit).
print("Platt a, b =", platt_a, platt_b)

In [ ]:
# --- vocab.json (word+char vocab + idf) and coef.json ---
word_vec = vectorizer.transformer_list[0][1]
char_vec = vectorizer.transformer_list[1][1]

vocab = {
    "description": "word block occupies indices [0, n_word); char block [n_word, n_word+n_char). Each block is L2-normalised INDEPENDENTLY, then concatenated.",
    "word_params": WORD_PARAMS, "char_params": CHAR_PARAMS,
    "n_word": len(word_vec.vocabulary_), "n_char": len(char_vec.vocabulary_),
    "n_features": n_features,
    "word": {"vocabulary": {k: int(v) for k, v in word_vec.vocabulary_.items()},
             "idf": word_vec.idf_.tolist()},
    "char": {"vocabulary": {k: int(v) for k, v in char_vec.vocabulary_.items()},
             "idf": char_vec.idf_.tolist()},
}
with open(os.path.join(OUT, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f)

coef = {
    "intercept": float(lr.intercept_[0]),
    "coef": lr.coef_[0].astype(float).tolist(),
    "calibration": {"method": "platt", "a": platt_a, "b": platt_b,
                    "formula": "p = 1/(1+exp(a*f + b)); f = intercept + sum(coef_i * tfidf_i)"},
    "labels": {"0": "legitimate", "1": "fraudulent"},
    "note": "coef_i * tfidf_i is a LOG-ODDS contribution, not a probability.",
}
with open(os.path.join(OUT, "coef.json"), "w") as f:
    json.dump(coef, f)
print("Wrote vocab.json (n_word=%d, n_char=%d) and coef.json" % (vocab["n_word"], vocab["n_char"]))

In [ ]:
# --- Fixed test vector + exact probability (the number Java asserts) ---
FIXED_TEXT = ("URGENT hiring! Work from home and earn $5000 per week, no experience needed. "
              "Send your bank account details and a $200 processing fee to start immediately.")
xv = vectorizer.transform([FIXED_TEXT])
served_p = float(served.predict_proba(xv)[0, 1])   # end-to-end calibrated probability
raw_p = float(lr.predict_proba(xv)[0, 1])           # raw LR positive-class probability

onnx_p = None
if classifier_onnx_ok:
    import onnxruntime as rt
    dense = xv.toarray().astype(np.float32)
    sess = rt.InferenceSession(os.path.join(OUT, "classifier.onnx"), providers=["CPUExecutionProvider"])
    outs = sess.run(None, {"input": dense})
    prob_out = next(o for o in outs if getattr(o, "ndim", 0) == 2 and o.shape[-1] == 2)
    onnx_p = float(prob_out[0, 1])
    sk_p = served_p if export_kind == "calibrated" else raw_p
    # ONNX runs float32, sklearn float64 over ~80k dims -> compare at 1e-4.
    # (The Java test asserts Java-ONNX == this onnx_probability_pos within 1e-6: same float32 runtime.)
    print(f"ONNX prob (pos):          {onnx_p:.10f}")
    print(f"sklearn matching prob:    {sk_p:.10f}")
    print(f"ABS DIFF (float32 vs 64): {abs(onnx_p - sk_p):.3e}   (expect < 1e-4)")
    assert abs(onnx_p - sk_p) < 1e-4, "ONNX/sklearn parity check failed"

print(f"RAW LR prob (pos):        {raw_p:.10f}")
print(f"SERVED calibrated prob:   {served_p:.10f}   <-- Java target for FIXED_TEXT")

nz = xv.tocoo()
fixed = {
    "fixed_text": FIXED_TEXT,
    "export_kind": export_kind,
    "input_dim": int(n_features),
    "nonzero_indices": [int(i) for i in nz.col],
    "nonzero_values": [float(v) for v in nz.data],
    "onnx_probability_pos": onnx_p,             # null if no ONNX; assert target for Java-ONNX
    "raw_lr_probability_pos": raw_p,
    "served_calibrated_probability": served_p,  # end-to-end target for FIXED_TEXT
}
with open(os.path.join(OUT, "fixed_test_vector.json"), "w") as f:
    json.dump(fixed, f)
print("Wrote fixed_test_vector.json  (nnz =", len(fixed["nonzero_indices"]), ")")

In [ ]:
# --- Salary regressor -> salary.onnx (re-fit from 03 logic) ---
sal = df[df["salary_range"].notna()].copy()
parts = sal["salary_range"].astype(str).str.split("-", n=1, expand=True)
sal["lo"] = pd.to_numeric(parts[0], errors="coerce")
sal["hi"] = pd.to_numeric(parts[1], errors="coerce")
sal = sal.dropna(subset=["lo", "hi"])
sal = sal[(sal["lo"] > 0) & (sal["hi"] >= sal["lo"])]
sal["mid"] = (sal["lo"] + sal["hi"]) / 2.0
sal = sal[(sal["mid"] >= 1) & (sal["mid"] <= 1_000_000)]
sal["log_pay"] = np.log1p(sal["mid"])
sal["country"] = sal["location"].apply(lambda s: (s.split(",")[0].strip().upper()[:2]
                                                  if isinstance(s, str) and s.strip() else "Unknown"))
for c in ["required_experience", "required_education", "employment_type"]:
    sal[c] = sal[c].fillna("Unknown").astype(str)
sal["title"] = sal["title"].fillna("").astype(str)
CAT_COLS = ["required_experience", "required_education", "employment_type", "country"]

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

pre = ColumnTransformer([
    ("cats", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ("title", TfidfVectorizer(max_features=300, ngram_range=(1, 1)), "title"),
])
salary_model = Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0, random_state=SEED))])
Xtr, Xva, ytr, yva = train_test_split(sal[CAT_COLS + ["title"]], sal["log_pay"].values,
                                      test_size=0.2, random_state=SEED)
salary_model.fit(Xtr, ytr)
resid_std = float(np.std(yva - salary_model.predict(Xva), ddof=1))

# ALWAYS write salary_params.json (guaranteed Java-loadable). skl2onnx export of a
# string ColumnTransformer is fragile, so ONNX is a best-effort bonus, not the contract.
ohe = salary_model.named_steps["pre"].named_transformers_["cats"]
tfv = salary_model.named_steps["pre"].named_transformers_["title"]
ridge = salary_model.named_steps["reg"]
salary_params = {
    "target": "log1p(midpoint of salary_range)",
    "residual_std_log": resid_std, "z_flag_threshold": 3.0,
    "cat_cols": CAT_COLS,
    "ohe_categories": [[str(v) for v in cats] for cats in ohe.categories_],
    "title_tfidf": {
        "vocabulary": {k: int(v) for k, v in tfv.vocabulary_.items()},
        "idf": tfv.idf_.tolist(),
        "params": {"max_features": 300, "ngram_range": [1, 1], "norm": "l2",
                   "sublinear_tf": False, "smooth_idf": True, "lowercase": True},
    },
    "ridge_coef": [float(c) for c in ridge.coef_],
    "ridge_intercept": float(ridge.intercept_),
    "feature_order": "one-hot(cat_cols in order; unknown category -> all-zeros) concatenated "
                     "with L2-normalized title TF-IDF; dot with ridge_coef then add intercept.",
}
with open(os.path.join(OUT, "salary_params.json"), "w", encoding="utf-8") as f:
    json.dump(salary_params, f)
with open(os.path.join(OUT, "salary_meta.json"), "w") as f:
    json.dump({"residual_std_log": resid_std, "z_flag_threshold": 3.0,
               "target": "log1p(midpoint)", "cat_cols": CAT_COLS}, f, indent=2)
print("Wrote salary_params.json (Java-loadable) + salary_meta.json. Residual std:", round(resid_std, 4))

try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import StringTensorType
    inits = [(c, StringTensorType([None, 1])) for c in CAT_COLS] + [("title", StringTensorType([None, 1]))]
    sal_onx = convert_sklearn(salary_model, initial_types=inits)
    with open(os.path.join(OUT, "salary.onnx"), "wb") as f:
        f.write(sal_onx.SerializeToString())
    print("Also wrote salary.onnx (bonus).")
except Exception as e:
    print("salary.onnx export skipped (string-pipeline fragility); Java uses salary_params.json:", repr(e))

In [ ]:
# --- Embed the 866 confirmed frauds -> known_scams seed SQL ---
from sentence_transformers import SentenceTransformer

frauds = df[df["fraudulent"] == 1].copy()
fraud_text = build_text(frauds)
emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embs = emb_model.encode(fraud_text.tolist(), batch_size=64, show_progress_bar=True,
                        normalize_embeddings=False)
print("Embedded", len(embs), "frauds -> dim", embs.shape[1])

def esc(s):
    return str(s).replace("'", "''")

seed_path = os.path.join(OUT, "known_scams_seed.sql")
with open(seed_path, "w", encoding="utf-8") as f:
    f.write("-- known_scams seed: EMSCAD confirmed frauds embedded with all-MiniLM-L6-v2 (384-d).\n")
    f.write("-- Load AFTER V1__init.sql. Generated by ml/notebooks/04_export_onnx.ipynb.\n")
    for txt, e in zip(fraud_text.tolist(), embs):
        vec = "[" + ",".join(f"{x:.6f}" for x in e) + "]"
        safe = esc(str(txt)[:20000])          # truncate FIRST, then escape, so '' pairs stay intact
        f.write("INSERT INTO known_scams (text, embedding, source, confirmed_at) VALUES "
                f"('{safe}', '{vec}'::vector, 'EMSCAD', now());\n")
print("Wrote", seed_path)

In [ ]:
# --- Manifest of everything produced ---
import glob as _g
print("Artifacts in", OUT, ":")
for p in sorted(_g.glob(os.path.join(OUT, "*"))):
    print(f"  {os.path.basename(p):28} {os.path.getsize(p):>12,} bytes")
print("\nCopy classifier.onnx, salary.onnx, vocab.json into backend/src/main/resources/models/")
print("(these are gitignored). fixed_test_vector.json drives the Java 1e-6 parity test.")

### The Java parity contract

- `classifier.onnx` takes the **exact** TF-IDF vector as input. The Java unit test should feed
  the vector reconstructed from `fixed_test_vector.json` (`nonzero_indices`/`nonzero_values`)
  and assert the output equals `onnx_probability_pos` within **1e-6** — this isolates ONNX
  runtime parity from the Java TF-IDF reimplementation.
- Java rebuilds the TF-IDF vector from `vocab.json` (a separate, looser-tolerance test), and
  computes contributions and the calibrated probability from `coef.json`.
- `served_calibrated_probability` is the end-to-end number for `FIXED_TEXT`.